# Recuperación de información en NFCorpus

## 1. Configuración

Se fijan el corpus, la partición de evaluación, el corte de los rankings y los parámetros de BM25. El notebook se ejecuta con un kernel de Python 3 y Node.js 20 o posterior disponible en el entorno. Python carga los datos y calcula las métricas. Node.js ejecuta el mismo motor léxico que utiliza la API.

La carpeta `datos` puede contener los archivos extraídos en `nfcorpus/` o el archivo `nfcorpus.zip` con la misma estructura interna.

In [1]:
from pathlib import Path
from shutil import which
from subprocess import run

DATOS = Path("datos")
NOMBRE_CORPUS = "nfcorpus"
CARPETA_CORPUS = DATOS / NOMBRE_CORPUS
CORPUS = CARPETA_CORPUS / "corpus.jsonl"
CONSULTAS = CARPETA_CORPUS / "queries.jsonl"
QRELS = CARPETA_CORPUS / "qrels/test.tsv"
ARCHIVO_DATOS = DATOS / f"{NOMBRE_CORPUS}.zip"
TOP_K = 10
K1, B = 1.5, 0.75

NODE = which("node")
if NODE is None:
    raise RuntimeError("Se requiere Node.js 20 o posterior para ejecutar el motor.")
version_node = run([NODE, "--version"], capture_output=True, text=True, check=True).stdout.strip()
if int(version_node[1:].split(".")[0]) < 20:
    raise RuntimeError(f"Se requiere Node.js 20 o posterior. Versión encontrada: {version_node}.")

print(f"Corpus: {NOMBRE_CORPUS} | Partición: {QRELS.stem} | Top {TOP_K}")
print(f"BM25: k1={K1}, b={B} | Node.js: {version_node}")

Corpus: nfcorpus | Partición: test | Top 10
BM25: k1=1.5, b=0.75 | Node.js: v25.9.0


## 2. Carga del corpus y selección de consultas

Cada registro se indexa como un documento completo a partir de su título y su texto. El campo `_id` de BEIR identifica el documento durante todo el proceso. Se comprueba que los identificadores sean únicos y que todos los juicios se refieran a consultas y documentos del corpus.

Se evalúan todas las consultas con juicios en `qrels/test.tsv`. El bloque cruza sus identificadores con `queries.jsonl` y deja registrada la lista utilizada.

In [2]:
import csv
import json
from collections import Counter, defaultdict
from contextlib import contextmanager
from io import TextIOWrapper
from itertools import islice
from zipfile import ZipFile


@contextmanager
def abrir_datos(ruta):
    if ruta.is_file():
        with ruta.open(encoding="utf-8", newline="") as archivo:
            yield archivo
        return

    if not ARCHIVO_DATOS.is_file():
        raise FileNotFoundError(f"Falta {ruta} o el archivo {ARCHIVO_DATOS}.")

    miembro = ruta.relative_to(DATOS).as_posix()
    with ZipFile(ARCHIVO_DATOS) as comprimido:
        with comprimido.open(miembro) as contenido:
            with TextIOWrapper(contenido, encoding="utf-8", newline="") as archivo:
                yield archivo


def leer_jsonl(ruta):
    with abrir_datos(ruta) as archivo:
        return [json.loads(linea) for linea in archivo if linea.strip()]


documentos = leer_jsonl(CORPUS)
consultas = leer_jsonl(CONSULTAS)
ids_documentos = {doc["_id"] for doc in documentos}
texto_por_id = {consulta["_id"]: consulta["text"] for consulta in consultas}

assert len(ids_documentos) == len(documentos)
assert len(texto_por_id) == len(consultas)
assert all(isinstance(doc["_id"], str) and doc["_id"] and doc["text"].strip() for doc in documentos)

juicios = defaultdict(dict)
with abrir_datos(QRELS) as archivo:
    for fila in csv.DictReader(archivo, delimiter="\t"):
        query_id, doc_id = fila["query-id"], fila["corpus-id"]
        grado = int(fila["score"])
        assert query_id in texto_por_id and doc_id in ids_documentos
        assert doc_id not in juicios[query_id] and grado >= 0
        juicios[query_id][doc_id] = grado

consultas_test = {query_id: texto_por_id[query_id] for query_id in sorted(juicios)}
assert consultas_test and all(texto.strip() for texto in consultas_test.values())
grados = Counter(grado for documentos_juzgados in juicios.values() for grado in documentos_juzgados.values())

print(f"Documentos: {len(documentos)} | Consultas test: {len(consultas_test)} | Juicios: {sum(grados.values())}")
print(f"Frecuencias de los grados de relevancia: {dict(sorted(grados.items()))}")
for query_id, texto in islice(consultas_test.items(), 5):
    print(f"{query_id:12} {' '.join(texto.split())}")
print("\nIDs usados en la evaluación")
ids_test = tuple(consultas_test)
for inicio in range(0, len(ids_test), 10):
    print(" ".join(ids_test[inicio:inicio + 10]))

Documentos: 3633 | Consultas test: 323 | Juicios: 12334
Frecuencias de los grados de relevancia: {1: 11758, 2: 576}
PLAIN-1008   deafness
PLAIN-1018   DHA
PLAIN-102    Stopping Heart Disease in Childhood
PLAIN-1028   dietary scoring
PLAIN-1039   domoic acid

IDs usados en la evaluación
PLAIN-1008 PLAIN-1018 PLAIN-102 PLAIN-1028 PLAIN-1039 PLAIN-1050 PLAIN-1066 PLAIN-1088 PLAIN-1098 PLAIN-1109
PLAIN-1119 PLAIN-112 PLAIN-1130 PLAIN-1141 PLAIN-1151 PLAIN-1161 PLAIN-1172 PLAIN-1183 PLAIN-1193 PLAIN-12
PLAIN-1203 PLAIN-1214 PLAIN-1225 PLAIN-123 PLAIN-1236 PLAIN-1249 PLAIN-1262 PLAIN-1275 PLAIN-1288 PLAIN-1299
PLAIN-1309 PLAIN-1320 PLAIN-133 PLAIN-1331 PLAIN-1342 PLAIN-1353 PLAIN-1363 PLAIN-1374 PLAIN-1387 PLAIN-1398
PLAIN-1409 PLAIN-1419 PLAIN-1429 PLAIN-143 PLAIN-1441 PLAIN-1453 PLAIN-1463 PLAIN-1473 PLAIN-1485 PLAIN-1496
PLAIN-1506 PLAIN-1516 PLAIN-1527 PLAIN-153 PLAIN-1537 PLAIN-1547 PLAIN-1557 PLAIN-1568 PLAIN-1579 PLAIN-1590
PLAIN-1601 PLAIN-1611 PLAIN-1621 PLAIN-1635 PLAIN-1645 PLAIN-

## 3. Preprocesamiento léxico

Los documentos y las consultas pasan por la misma secuencia. Se normaliza el texto con Unicode NFKC y se convierte a minúsculas. Se recorta la terminación de contracciones inglesas como `n't`. La tokenización conserva secuencias de letras y números y toma la puntuación como separador. Después se eliminan palabras frecuentes del inglés y se aplican reglas de stemming basadas en Porter a los tokens de letras de la a a la z.

El ejemplo muestra la eliminación de palabras frecuentes y la reducción de `diets` a `diet`. El número `25` se conserva.

In [3]:
FUENTE_PREPROCESO = r"""
// Preprocesamiento común para documentos y consultas en inglés.
const stopwords = new Set(`a about above after again against all am an and any are as at be
because been before being below between both but by can could d did do does doing down
during each few for from further had has have having he her here hers herself him himself
his how i if in into is it its itself just ll m me might more most must my myself no nor not
now of off on once only or other ought our ours ourselves out over own re s same she should
so some such t than that the their theirs them themselves then there these they this those
through to too under until up ve very was we were what when where which while who whom why
will with would you your yours yourself yourselves`.split(/\s+/u));

const paso2 = {
  ational: 'ate', tional: 'tion', enci: 'ence', anci: 'ance', izer: 'ize',
  bli: 'ble', alli: 'al', entli: 'ent', eli: 'e', ousli: 'ous', ization: 'ize',
  ation: 'ate', ator: 'ate', alism: 'al', iveness: 'ive', fulness: 'ful',
  ousness: 'ous', aliti: 'al', iviti: 'ive', biliti: 'ble', logi: 'log',
};
const paso3 = {
  icate: 'ic', ative: '', alize: 'al', iciti: 'ic', ical: 'ic', ful: '', ness: '',
};
const paso4 = Object.fromEntries(
  'al ance ence er ic able ible ant ement ment ent ion ou ism ate iti ous ive ize'
    .split(' ').map(sufijo => [sufijo, '']),
);

function estructura(palabra) {
  let patron = '';
  for (const letra of palabra) {
    const vocal = 'aeiou'.includes(letra) || (letra === 'y' && patron.endsWith('C'));
    patron += vocal ? 'V' : 'C';
  }
  return patron;
}

function medida(palabra) {
  return (estructura(palabra).match(/VC/g) ?? []).length;
}

function terminaCVC(palabra) {
  return estructura(palabra).endsWith('CVC') && !/[wxy]$/u.test(palabra);
}

function ordenarSufijos(reglas) {
  return Object.entries(reglas).sort(([a], [b]) => b.length - a.length);
}

const reglas2 = ordenarSufijos(paso2);
const reglas3 = ordenarSufijos(paso3);
const reglas4 = ordenarSufijos(paso4);

function aplicarSufijo(palabra, reglas, medidaMinima) {
  for (const [sufijo, reemplazo] of reglas) {
    if (!palabra.endsWith(sufijo)) continue;
    const raiz = palabra.slice(0, -sufijo.length);
    if (sufijo === 'ion' && !/[st]$/u.test(raiz)) return palabra;
    return medida(raiz) >= medidaMinima ? raiz + reemplazo : palabra;
  }
  return palabra;
}

function porter(palabra) {
  if (palabra.length <= 2 || !/^[a-z]+$/u.test(palabra)) return palabra;
  palabra = palabra.replace(/sses$/u, 'ss').replace(/ies$/u, 'i');
  if (!palabra.endsWith('ss')) palabra = palabra.replace(/s$/u, '');

  if (palabra.endsWith('eed')) {
    if (medida(palabra.slice(0, -3)) > 0) palabra = palabra.slice(0, -1);
  } else {
    const raiz = palabra.replace(/(?:ed|ing)$/u, '');
    if (raiz !== palabra && estructura(raiz).includes('V')) {
      palabra = raiz;
      if (/(at|bl|iz)$/u.test(palabra)) {
        palabra += 'e';
      } else if (/(.)\1$/u.test(palabra) && estructura(palabra).endsWith('C')) {
        if (!/[lsz]$/u.test(palabra)) palabra = palabra.slice(0, -1);
      } else if (medida(palabra) === 1 && terminaCVC(palabra)) {
        palabra += 'e';
      }
    }
  }

  if (palabra.endsWith('y') && estructura(palabra.slice(0, -1)).includes('V')) {
    palabra = palabra.slice(0, -1) + 'i';
  }
  palabra = aplicarSufijo(palabra, reglas2, 1);
  palabra = aplicarSufijo(palabra, reglas3, 1);
  palabra = aplicarSufijo(palabra, reglas4, 2);

  if (palabra.endsWith('e')) {
    const raiz = palabra.slice(0, -1);
    const m = medida(raiz);
    if (m > 1 || (m === 1 && !terminaCVC(raiz))) palabra = raiz;
  }
  if (palabra.endsWith('ll') && medida(palabra) > 1) palabra = palabra.slice(0, -1);
  return palabra;
}

export function tokenizar(texto, cache) {
  const normalizado = texto.normalize('NFKC').toLowerCase().replace(/n['’]t\b/gu, '');
  const tokens = normalizado.match(/[\p{L}\p{N}]+/gu) ?? [];
  return tokens.filter(token => !stopwords.has(token)).map(token => {
    if (!cache) return porter(token);
    if (!cache.has(token)) cache.set(token, porter(token));
    return cache.get(token);
  });
}
"""


def ejecutar_motor(fuente, instrucciones, entrada):
    programa = (
        'import { readFileSync } from "node:fs";\n'
        'const entrada = JSON.parse(readFileSync(0, "utf8"));\n'
        + fuente + "\n" + instrucciones
    )
    proceso = run(
        [NODE, "--input-type=module", "-e", programa],
        input=json.dumps(entrada, ensure_ascii=False),
        encoding="utf-8",
        capture_output=True,
        timeout=90,
    )
    if proceso.returncode:
        raise RuntimeError(proceso.stderr.strip())
    return json.loads(proceso.stdout)


ejemplo = "The effects of diets and dietary changes on 25 studies."
print(f"Texto original = {ejemplo}")
print("Tokens =", ejecutar_motor(FUENTE_PREPROCESO, "console.log(JSON.stringify(tokenizar(entrada)));", ejemplo))

Texto original = The effects of diets and dietary changes on 25 studies.
Tokens = ['effect', 'diet', 'dietari', 'chang', '25', 'studi']


## 4. Índice invertido y modelos de recuperación

El vocabulario contiene los términos que sobreviven al preprocesamiento. Para cada término se guardan las posiciones de los documentos donde aparece y su frecuencia dentro de cada documento. La cantidad de pares da la frecuencia documental $df$. Cada posición permite recuperar el identificador original, el título y un extracto del texto para la respuesta.

- **TF-IDF con similitud coseno.** El peso de un término es $(1+\ln tf)\,\ln(1+N/df)$ tanto en documentos como en consultas. $N$ es el total de documentos y $tf$ es la frecuencia del término en el texto que se representa. El coseno divide el producto escalar por la norma L2 de cada vector. La norma de la consulta se calcula sobre los términos que pertenecen al vocabulario.
- **BM25.** Cada término distinto de la consulta aporta $\ln(1+(N-df+0.5)/(df+0.5))\,\frac{tf(k_1+1)}{tf+k_1(1-b+b|d|/\overline{|d|})}$. La longitud $|d|$ es el número de tokens del documento después del preprocesamiento. El parámetro $k_1$ controla cuánto aporta la repetición del término y $b$ ajusta el efecto de la longitud.

Cada método puntúa los documentos que comparten términos con la consulta. Los empates se ordenan por identificador y se devuelven hasta `top_k` coincidencias. El bloque construye el índice y muestra los identificadores y frecuencias asociados a un término.

In [4]:
FUENTE_INDICE = r"""
// Vocabulario, frecuencias y estadísticas de los documentos originales.
function contar(terminos) {
  const frecuencias = new Map();
  for (const termino of terminos) {
    frecuencias.set(termino, (frecuencias.get(termino) ?? 0) + 1);
  }
  return frecuencias;
}

function fragmento(texto) {
  const limpio = texto.trim().replace(/\s+/gu, ' ');
  return limpio.length <= 200 ? limpio : `${limpio.slice(0, 200)}…`;
}

export function construirIndice(corpus, k1 = 1.5, b = 0.75) {
  if (!corpus.length) throw new Error('El corpus está vacío.');
  const invertido = new Map();
  const cache = new Map();
  const longitudes = [];
  const documentos = corpus.map((doc, posicion) => {
    const frecuencias = contar(tokenizar(`${doc.title ?? ''} ${doc.text}`, cache));
    let longitud = 0;
    for (const [termino, tf] of frecuencias) {
      if (!invertido.has(termino)) invertido.set(termino, []);
      invertido.get(termino).push(posicion, tf);
      longitud += tf;
    }
    longitudes.push(longitud);
    return { id: doc._id, title: doc.title?.trim() ?? '', snippet: fragmento(doc.text) };
  });

  const n = documentos.length;
  const longitudMedia = longitudes.reduce((total, longitud) => total + longitud, 0) / n;
  const normasCuadradas = Array(n).fill(0);
  const vocabulario = Object.create(null);
  for (const [termino, postings] of invertido) {
    const df = postings.length / 2;
    const idfTfidf = Math.log1p(n / df);
    const idfBm25 = Math.log1p((n - df + 0.5) / (df + 0.5));
    vocabulario[termino] = [idfTfidf, idfBm25, postings];
    for (let i = 0; i < postings.length; i += 2) {
      const peso = (1 + Math.log(postings[i + 1])) * idfTfidf;
      normasCuadradas[postings[i]] += peso ** 2;
    }
  }

  const normasTfidf = normasCuadradas.map(Math.sqrt);
  const normasBm25 = longitudes.map(longitud =>
    k1 * (1 - b + b * (longitudMedia ? longitud / longitudMedia : 0)),
  );
  return { documentos, vocabulario, longitudMedia, normasTfidf, normasBm25, k1 };
}

// Ranking de los documentos con términos en común y desempate por ID original.
export function buscar(indice, consulta, modelo, topK) {
  const frecuencias = contar(tokenizar(consulta));
  const scores = new Map();
  let normaConsultaCuadrada = 0;

  for (const [termino, tfConsulta] of frecuencias) {
    if (!Object.hasOwn(indice.vocabulario, termino)) continue;
    const [idfTfidf, idfBm25, postings] = indice.vocabulario[termino];
    const pesoConsulta = (1 + Math.log(tfConsulta)) * idfTfidf;
    if (modelo === 'tfidf') normaConsultaCuadrada += pesoConsulta ** 2;

    for (let i = 0; i < postings.length; i += 2) {
      const posicion = postings[i];
      const tf = postings[i + 1];
      const aporte = modelo === 'tfidf'
        ? pesoConsulta * (1 + Math.log(tf)) * idfTfidf
        : idfBm25 * tf * (indice.k1 + 1) / (tf + indice.normasBm25[posicion]);
      scores.set(posicion, (scores.get(posicion) ?? 0) + aporte);
    }
  }

  if (modelo === 'tfidf') {
    const normaConsulta = Math.sqrt(normaConsultaCuadrada);
    for (const [posicion, score] of scores) {
      const coseno = score / (normaConsulta * indice.normasTfidf[posicion]);
      scores.set(posicion, Math.min(1, coseno));
    }
  }

  const documentos = indice.documentos;
  return [...scores].sort(([a, scoreA], [b, scoreB]) => {
    if (scoreA !== scoreB) return scoreB - scoreA;
    return documentos[a].id < documentos[b].id ? -1 : 1;
  }).slice(0, topK).map(([posicion, score], i) => ({
    rank: i + 1,
    document_id: documentos[posicion].id,
    score,
    title: documentos[posicion].title,
    snippet: documentos[posicion].snippet,
  }));
}
"""


FUENTE_MOTOR = FUENTE_PREPROCESO + FUENTE_INDICE
entrada_indice = {"corpus": documentos, "k1": K1, "b": B}
estadisticas = ejecutar_motor(FUENTE_MOTOR, """
    const indice = construirIndice(entrada.corpus, entrada.k1, entrada.b);
    const termino = Object.hasOwn(indice.vocabulario, 'diet')
        ? 'diet'
        : Object.keys(indice.vocabulario)[0];
    const postings = indice.vocabulario[termino]?.[2] ?? [];
    const muestra = [];
    for (let i = 0; i < Math.min(8, postings.length); i += 2) {
        muestra.push([indice.documentos[postings[i]].id, postings[i + 1]]);
    }
    console.log(JSON.stringify({
        vocabulario: Object.keys(indice.vocabulario).length,
        longitudMedia: indice.longitudMedia, termino, df: postings.length / 2, muestra,
    }));
""", entrada_indice)
print(f"Vocabulario: {estadisticas['vocabulario']} términos")
print(f"Longitud media: {estadisticas['longitudMedia']:.2f} tokens por documento")
print(f"Término {estadisticas['termino']!r} presente en {int(estadisticas['df'])} documentos")
print(f"Muestra de pares (ID, frecuencia) = {estadisticas['muestra']}")

Vocabulario: 19061 términos
Longitud media: 165.65 tokens por documento
Término 'diet' presente en 1041 documentos
Muestra de pares (ID, frecuencia) = [['MED-332', 1], ['MED-557', 1], ['MED-706', 1], ['MED-711', 6]]


## 5. Evaluación en la partición `test`

Los dos modelos reciben las mismas consultas y se evalúan a la profundidad 10. El bloque comprueba la validez de los identificadores, el orden de las puntuaciones y el tamaño de cada ranking antes de calcular las métricas. La tabla muestra la media aritmética de cada métrica por consulta, de modo que todas tienen el mismo peso.

- **Precision@10** divide el número de documentos relevantes recuperados por 10, incluso cuando se devuelven menos resultados.
- **Recall@10** divide ese número por el total de documentos relevantes registrados en los qrels de la consulta. Para precisión y exhaustividad, ambos grados, 1 y 2, cuentan como relevantes.
- **nDCG@10** conserva los grados. En la posición $i$, el aporte es $\text{grado}/\log_2(i+1)$, con una ganancia lineal como en `trec_eval`. Se divide la suma por el valor que se obtendría al colocar los documentos con mayor grado en las primeras posiciones. Los documentos sin juicio aportan cero.

Se reutilizan las funciones de precisión y exhaustividad del proyecto FiQA y se añade el cálculo de nDCG para aprovechar los grados de NFCorpus.

In [5]:
from collections.abc import Collection
from math import isfinite, log2
from statistics import mean

modelos = ("tfidf", "bm25")
rankings_puntuados = ejecutar_motor(FUENTE_MOTOR, """
    const indice = construirIndice(entrada.corpus, entrada.k1, entrada.b);
    const rankings = {};
    for (const [id, consulta] of Object.entries(entrada.consultas)) {
        rankings[id] = Object.fromEntries(entrada.modelos.map(modelo => [
            modelo, buscar(indice, consulta, modelo, entrada.k).map(({document_id, score}) => [document_id, score]),
        ]));
    }
    console.log(JSON.stringify(rankings));
""", {**entrada_indice, "consultas": consultas_test, "modelos": modelos, "k": TOP_K})


def _prepare(ranking: list[str], relevant: Collection[str], k: int) -> set[str]:
    """Valida ranking y k y devuelve los relevantes como conjunto."""
    if k <= 0:
        raise ValueError(f"k debe ser positivo, se recibió {k}")
    if len(set(ranking)) != len(ranking):
        raise ValueError("el ranking contiene doc_id duplicados")
    return set(relevant)


def _hits_at_k(ranking: list[str], relevant: set[str], k: int) -> int:
    return sum(1 for doc_id in ranking[:k] if doc_id in relevant)


def precision_at_k(ranking: list[str], relevant: Collection[str], k: int = 10) -> float:
    relevant = _prepare(ranking, relevant, k)
    # Se divide siempre por k: un ranking con menos de k documentos se
    # penaliza como si las posiciones faltantes fueran no relevantes.
    return _hits_at_k(ranking, relevant, k) / k


def recall_at_k(ranking: list[str], relevant: Collection[str], k: int = 10) -> float:
    relevant = _prepare(ranking, relevant, k)
    if not relevant:
        return 0.0
    # Se divide por todos los relevantes de la consulta, no solo por los
    # recuperados en el top-k.
    return _hits_at_k(ranking, relevant, k) / len(relevant)


def dcg(grados):
    return sum(grado / log2(posicion + 1) for posicion, grado in enumerate(grados, start=1))


def medir(ranking, qrels, k):
    relevantes = {doc_id for doc_id, grado in qrels.items() if grado > 0}
    grados_recuperados = [qrels.get(doc_id, 0) for doc_id in ranking[:k]]
    ideal = dcg(sorted(qrels.values(), reverse=True)[:k])
    return (
        precision_at_k(ranking, relevantes, k),
        recall_at_k(ranking, relevantes, k),
        dcg(grados_recuperados) / ideal if ideal else 0.0,
    )


resultados = {modelo: {} for modelo in modelos}
rankings = {}
assert set(rankings_puntuados) == set(consultas_test)
for query_id, ranking_por_modelo in rankings_puntuados.items():
    assert set(ranking_por_modelo) == set(modelos)
    rankings[query_id] = {}
    for modelo, filas in ranking_por_modelo.items():
        ranking = [doc_id for doc_id, _ in filas]
        scores = [score for _, score in filas]
        assert len(ranking) <= TOP_K and set(ranking) <= ids_documentos
        assert all(isfinite(score) for score in scores)
        assert scores == sorted(scores, reverse=True)
        rankings[query_id][modelo] = ranking
        resultados[modelo][query_id] = medir(ranking, juicios[query_id], TOP_K)

resumen = {
    modelo: tuple(mean(columna) for columna in zip(*por_consulta.values()))
    for modelo, por_consulta in resultados.items()
}
etiquetas = (f"Precision@{TOP_K}", f"Recall@{TOP_K}", f"nDCG@{TOP_K}")
print(f"{'Modelo':<10}" + "".join(f"{nombre:>15}" for nombre in etiquetas))
for modelo, metricas in resumen.items():
    print(f"{modelo:<10}" + "".join(f"{valor:15.5f}" for valor in metricas))

print("\nEjemplos por consulta")
print(f"{'ID':<12}{'Modelo':<10}" + "".join(f"{nombre:>15}" for nombre in etiquetas))
for query_id in islice(consultas_test, 5):
    for modelo in modelos:
        valores = resultados[modelo][query_id]
        print(f"{query_id:<12}{modelo:<10}" + "".join(f"{valor:15.5f}" for valor in valores))

Modelo       Precision@10      Recall@10        nDCG@10
tfidf             0.23065        0.15245        0.31998
bm25              0.24087        0.15502        0.33000

Ejemplos por consulta
ID          Modelo       Precision@10      Recall@10        nDCG@10
PLAIN-1008  tfidf             0.00000        0.00000        0.00000
PLAIN-1008  bm25              0.00000        0.00000        0.00000
PLAIN-1018  tfidf             0.50000        0.08333        0.56704
PLAIN-1018  bm25              0.60000        0.10000        0.52110
PLAIN-102   tfidf             0.00000        0.00000        0.00000
PLAIN-102   bm25              0.10000        0.04167        0.12531
PLAIN-1028  tfidf             0.00000        0.00000        0.00000
PLAIN-1028  bm25              0.00000        0.00000        0.00000
PLAIN-1039  tfidf             0.20000        1.00000        1.00000
PLAIN-1039  bm25              0.20000        1.00000        1.00000


### Comparación de los modelos

Se calcula la diferencia de las medias en puntos porcentuales y se cuenta en cuántas consultas gana cada modelo según el nDCG. El ejemplo compara los grados por posición en una consulta donde BM25 recupera más documentos relevantes y TF-IDF obtiene un nDCG mayor.

In [6]:
print("Diferencia de BM25 frente a TF-IDF en puntos porcentuales")
for nombre, bm25, tfidf in zip(etiquetas, resumen["bm25"], resumen["tfidf"]):
    print(f"{nombre:<13} {(bm25 - tfidf) * 100:+.2f}")

comparacion_ndcg = Counter()
for query_id in consultas_test:
    bm25 = resultados["bm25"][query_id][2]
    tfidf = resultados["tfidf"][query_id][2]
    if abs(bm25 - tfidf) < 1e-12:
        comparacion_ndcg["empate"] += 1
    else:
        comparacion_ndcg["bm25" if bm25 > tfidf else "tfidf"] += 1
print("Consultas con mayor nDCG:", dict(comparacion_ndcg))

caso = next((
    query_id
    for query_id in consultas_test
    if resultados["bm25"][query_id][0] > resultados["tfidf"][query_id][0]
    and resultados["bm25"][query_id][2] < resultados["tfidf"][query_id][2]
), None)
if caso is not None:
    print(f"\nGrados por posición para {caso} ({consultas_test[caso]})")
    print(f"{'Posición':>8} {'TF-IDF':>16} {'BM25':>16}")
    for posicion in range(TOP_K):
        celdas = []
        for modelo in modelos:
            ranking = rankings[caso][modelo]
            if posicion < len(ranking):
                doc_id = ranking[posicion]
                celdas.append(f"{doc_id} ({juicios[caso].get(doc_id, 0)})")
            else:
                celdas.append("—")
        print(f"{posicion + 1:>8} {celdas[0]:>16} {celdas[1]:>16}")

Diferencia de BM25 frente a TF-IDF en puntos porcentuales
Precision@10  +1.02
Recall@10     +0.26
nDCG@10       +1.00
Consultas con mayor nDCG: {'empate': 170, 'tfidf': 55, 'bm25': 98}

Grados por posición para PLAIN-1018 (DHA)
Posición           TF-IDF             BM25
       1     MED-4936 (1)     MED-5095 (0)
       2      MED-839 (0)     MED-4936 (1)
       3     MED-3012 (1)      MED-839 (0)
       4     MED-5095 (0)     MED-5091 (1)
       5     MED-5091 (1)      MED-928 (0)
       6     MED-5092 (1)     MED-3012 (1)
       7     MED-5093 (1)     MED-1832 (1)
       8     MED-5364 (0)     MED-5092 (1)
       9     MED-4736 (0)     MED-5093 (1)
      10      MED-928 (0)      MED-838 (0)


### Interpretación

BM25 obtuvo las tres medias más altas. Frente a TF-IDF, ganó 1,02 puntos porcentuales en precisión, 0,26 en exhaustividad y 1,00 en nDCG. En promedio se recuperaron 2,41 documentos relevantes entre los diez primeros con BM25 y 2,31 con TF-IDF. Para el nDCG por consulta, BM25 ganó en 98 casos, TF-IDF en 55 y hubo 170 empates.

En la consulta `PLAIN-1018` (DHA), BM25 encontró seis documentos con juicio positivo y TF-IDF encontró cinco. TF-IDF obtuvo un nDCG mayor porque colocó varios documentos con grado 1 en posiciones más altas. El primer resultado de TF-IDF tuvo grado 1 y el de BM25 aportó cero. La mejora media de BM25 describe el desempeño de estos modelos sobre las 323 consultas de NFCorpus al corte 10.

## 6. Prueba del servicio

`GET /health` comprueba que el índice está cargado. `POST /search` recibe `query`, `model` y `top_k`. La respuesta incluye el modelo y un ranking con posición, ID original, puntuación, título y extracto.

Esta celda ejecuta el manejador de la API en memoria. Se envía una consulta de texto libre a ambos modelos con `top_k=3` y se comprueban el estado, el orden, la ausencia de duplicados y los identificadores. También se comprueba el ranking vacío para una consulta sin coincidencias y el rechazo de un corte inválido.

In [7]:
FUENTE_API = r"""
const MAX_JSON_BYTES = 32_768;
const MAX_QUERY_LENGTH = 4_096;

function json(objeto, status = 200, headers = {}) {
  return Response.json(objeto, { status, headers });
}

async function leerJson(request) {
  if (!request.body) return { error: 'Envía un cuerpo JSON.', status: 400 };
  const reader = request.body.getReader();
  const decoder = new TextDecoder('utf-8', { fatal: true });
  let delCuerpo = 0;
  let texto = '';

  try {
    while (true) {
      const { value, done } = await reader.read();
      if (done) break;
      delCuerpo += value.byteLength;
      if (delCuerpo > MAX_JSON_BYTES) {
        await reader.cancel();
        return { error: 'El cuerpo supera 32 KiB.', status: 413 };
      }
      texto += decoder.decode(value, { stream: true });
    }
    return { datos: JSON.parse(texto + decoder.decode()) };
  } catch {
    return { error: 'El cuerpo debe ser JSON UTF-8 válido.', status: 400 };
  } finally {
    reader.releaseLock();
  }
}

export function crearServicio(cargarIndice, buscar) {
  return {
    async fetch(request) {
      const ruta = new URL(request.url).pathname;
      if (ruta === '/health' || ruta === '/') {
        if (request.method !== 'GET') {
          return json({ error: 'Usa GET.' }, 405, { Allow: 'GET' });
        }
        const indice = await cargarIndice();
        return json({
          status: 'ok', corpus: 'NFCorpus', documents: indice.documentos.length,
          source: 'https://huggingface.co/datasets/BeIR/nfcorpus',
          license: 'https://creativecommons.org/licenses/by-sa/4.0/',
        });
      }
      if (ruta !== '/search') return json({ error: 'Ruta no encontrada.' }, 404);
      if (request.method !== 'POST') {
        return json({ error: 'Usa POST.' }, 405, { Allow: 'POST' });
      }
      const tipo = request.headers.get('content-type')?.split(';')[0].trim().toLowerCase();
      if (tipo !== 'application/json') {
        return json({ error: 'Usa Content-Type: application/json.' }, 415);
      }

      const { datos, error, status } = await leerJson(request);
      if (error) return json({ error }, status);
      const { query, model, top_k } = datos ?? {};
      if (typeof query !== 'string' || !query.trim() || query.length > MAX_QUERY_LENGTH) {
        return json({ error: 'query debe contener entre 1 y 4096 caracteres.' }, 400);
      }
      if (model !== 'tfidf' && model !== 'bm25') {
        return json({ error: 'model debe ser tfidf o bm25.' }, 400);
      }
      if (!Number.isSafeInteger(top_k) || top_k < 1) {
        return json({ error: 'top_k debe ser un entero positivo.' }, 400);
      }

      const indice = await cargarIndice();
      return json({ model, results: buscar(indice, query, model, top_k) });
    },
  };
}
"""


consulta_libre = "plant-based food and cardiovascular health"
assert consulta_libre not in texto_por_id.values()
prueba_api = ejecutar_motor(FUENTE_MOTOR + FUENTE_API, """
    const indice = construirIndice(entrada.corpus, entrada.k1, entrada.b);
    const app = crearServicio(() => indice, buscar);
    async function pedirBusqueda(query, model, top_k) {
        const response = await app.fetch(new Request('http://localhost/search', {
            method: 'POST', headers: {'Content-Type': 'application/json'},
            body: JSON.stringify({query, model, top_k}),
        }));
        return { status: response.status, ...await response.json() };
    }

    const health = await app.fetch(new Request('http://localhost/health'));
    const busquedas = [];
    for (const model of ['tfidf', 'bm25']) {
        busquedas.push(await pedirBusqueda(entrada.query, model, 3));
    }
    const vacia = await pedirBusqueda('the and of', 'bm25', 3);
    const invalida = await pedirBusqueda(entrada.query, 'tfidf', 0);
    console.log(JSON.stringify({
        salud: { ...await health.json(), http_status: health.status },
        busquedas, vacia, invalida,
    }));
""", {**entrada_indice, "query": consulta_libre})

salud = prueba_api["salud"]
assert salud["http_status"] == 200 and salud["status"] == "ok"
assert salud["documents"] == len(documentos)
print(f"GET /health → 200 · {salud['corpus']} · {salud['documents']} documentos")
print(f"Consulta libre = {consulta_libre}")
for respuesta, modelo in zip(prueba_api["busquedas"], modelos, strict=True):
    assert respuesta["status"] == 200 and respuesta["model"] == modelo
    hallados = respuesta["results"]
    assert len(hallados) == 3 and [fila["rank"] for fila in hallados] == [1, 2, 3]
    assert len({fila["document_id"] for fila in hallados}) == 3
    scores = [fila["score"] for fila in hallados]
    assert scores == sorted(scores, reverse=True) and all(isfinite(score) for score in scores)
    print(f"\nPOST /search · {modelo} · top_k=3 → 200")
    for fila in hallados:
        assert fila["document_id"] in ids_documentos and isinstance(fila["title"], str)
        assert isinstance(fila["snippet"], str) and fila["snippet"].strip()
        print(f"{fila['rank']}. {fila['document_id']}  {fila['score']:.5f}  {fila['title']}")

assert prueba_api["vacia"]["status"] == 200 and prueba_api["vacia"]["results"] == []
assert prueba_api["invalida"]["status"] == 400 and prueba_api["invalida"]["error"]
print("\nConsulta sin coincidencias → 200 con lista vacía")
print("Corte top_k=0 → 400")

GET /health → 200 · NFCorpus · 3633 documentos
Consulta libre = plant-based food and cardiovascular health

POST /search · tfidf · top_k=3 → 200
1. MED-4349  0.26680  Anti-inflammatory effects of plant-based foods and of their constituents.
2. MED-1991  0.22557  Plant foods and plant-based diets: protective against childhood obesity?
3. MED-4831  0.21759  Effects of plant-based diets on plasma lipids.

POST /search · bm25 · top_k=3 → 200
1. MED-4349  15.23581  Anti-inflammatory effects of plant-based foods and of their constituents.
2. MED-1991  13.22866  Plant foods and plant-based diets: protective against childhood obesity?
3. MED-1489  12.82099  A way to reverse CAD?

Consulta sin coincidencias → 200 con lista vacía
Corte top_k=0 → 400
